In [9]:
from pathlib import Path

ROOT_DIR = Path(r"F:\trainingAggression_MR\maladaptive")

leaf_folders_without_avi = []

for folder in ROOT_DIR.rglob("*"):
    if not folder.is_dir():
        continue

    # Check if this is a leaf folder (contains no subdirectories)
    if any(item.is_dir() for item in folder.iterdir()):
        continue

    # Check for .avi files (case-insensitive)
    has_avi = any(
        item.is_file() and item.suffix.lower() == ".avi"
        for item in folder.iterdir()
    )

    if not has_avi:
        leaf_folders_without_avi.append(folder)

print(f"Found {len(leaf_folders_without_avi)} leaf folders without .avi files:\n")

for folder in leaf_folders_without_avi:
    print(folder)

Found 0 leaf folders without .avi files:



In [11]:
from pathlib import Path
import subprocess
import shlex
import re
import time


# ============================================================
# CONFIGURATION
# ============================================================

# Root containing all experimental groups / mouse folders
ROOT_DIR = Path(
    r"A:\rawData\raw_trainingSessions\Maladaptive\Psilocybin\20260825_mouse1089289_trainingSessions\Day03"
)

# Compressed videos are written here
OUTPUT_ROOT = Path(
    r"C:\Users\stagk\OneDrive\Desktop\lol"
)

# FFmpeg executable
FFMPEG = (r"C:\Users\stagk\OneDrive\Desktop\ffmpeg\bin\ffmpeg.exe"
)

# Encoding settings
PRESET = "slow"

# CRF values
CRF_MINISCOPE = 22
CRF_OPTOGENETICS = 16

# Target resolutions matching Optogenetics videos
FRONT_WIDTH = 592
FRONT_HEIGHT = 240

TOP_WIDTH = 552
TOP_HEIGHT = 292

# If True, FFmpeg receives -y
OVERWRITE = True

# If True, commands are printed but not executed
DRY_RUN = False


# ============================================================
# PATH / VIDEO TYPE HELPERS
# ============================================================

def find_mouse_id_from_path(path: Path) -> str:
    """
    Find mouse ID anywhere in the path.

    Example:
        20260126_mouse1034814_trainingSessions

    Returns:
        mouse1034814
    """

    for part in path.parts:

        match = re.search(
            r"(mouse\d+)",
            part,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1)

    return "unknownMouse"


def find_day_path(path: Path):
    """
    Find the nearest DayXX directory above the AVI.

    Examples:
        Day01
        Day02
        Day03
        Day10
    """

    for parent in path.parents:

        if re.fullmatch(
            r"day\d+",
            parent.name,
            flags=re.IGNORECASE
        ):
            return parent

    return None


def detect_view_from_path(path: Path) -> str:
    """
    Detect FrontView or TopView from anywhere in the
    directory hierarchy.

    Handles names such as:

        Miniscope(FrontView)
        Miniscope(TopView)
        Optogenetics(FrontView)
        Optogenetics(TopView)
        FrontView
        TopView
    """

    # First look for explicit names
    for part in reversed(path.parts):

        name = part.lower()

        if (
            "topview" in name
            or "top_view" in name
            or "top view" in name
        ):
            return "topView"

        if (
            "frontview" in name
            or "front_view" in name
            or "front view" in name
        ):
            return "frontView"

    # More tolerant fallback
    for part in reversed(path.parts):

        name = part.lower()

        if "top" in name:
            return "topView"

        if "front" in name:
            return "frontView"

    # Default if nothing identifiable exists
    return "frontView"


def is_optogenetic(path: Path) -> bool:
    """
    True if 'optogen' occurs anywhere in the path.

    This catches:
        Optogenetics
        Optogenetic
        optogen
        etc.
    """

    return "optogen" in str(path).lower()


def is_miniscope(path: Path) -> bool:
    """
    True if 'miniscope' occurs anywhere in the path.
    """

    return "miniscope" in str(path).lower()


def is_already_compressed_input(path: Path) -> bool:
    """
    Skip input files already ending in _comp.avi.
    """

    return path.name.lower().endswith("_comp.avi")


def output_already_exists(path: Path) -> bool:
    """
    Skip compression when expected output already exists.
    """

    return path.exists()


# ============================================================
# RESOLUTION / FILTER SETTINGS
# ============================================================

def get_miniscope_filter(view: str) -> str:
    """
    Create FFmpeg scaling filter for Miniscope videos.

    Important:
    - preserves aspect ratio
    - NEVER stretches the image
    - downsamples using AREA interpolation
    - pads to the exact target resolution
    - does not crop behavioral information

    Targets:

        FrontView = 592 x 240
        TopView   = 552 x 292
    """

    if view == "topView":

        width = TOP_WIDTH
        height = TOP_HEIGHT

    else:

        width = FRONT_WIDTH
        height = FRONT_HEIGHT

    video_filter = (
        f"scale="
        f"{width}:{height}:"
        f"force_original_aspect_ratio=decrease:"
        f"flags=area,"
        f"pad="
        f"{width}:{height}:"
        f"(ow-iw)/2:"
        f"(oh-ih)/2"
    )

    return video_filter


def add_view_rotation_filter(video_filter, view: str):
    """
    Rotate every TopView video by 180 degrees.

    FFmpeg implements a 180-degree rotation as:
        hflip,vflip

    This is applied to ALL TopView videos, regardless of whether
    they are Miniscope, Optogenetics, or another video type.

    If another filter already exists (for example Miniscope
    resize/pad), the rotation is appended to that filter chain.
    """

    if view != "topView":
        return video_filter

    rotation_filter = "hflip,vflip"

    if video_filter is None:
        return rotation_filter

    return f"{video_filter},{rotation_filter}"


# ============================================================
# FFMPEG
# ============================================================

def compress_with_ffmpeg(
    src: Path,
    dst: Path,
    crf: int,
    video_filter=None,
    preset: str = PRESET,
    overwrite: bool = OVERWRITE
):
    """
    Compress a single AVI using H.264 / libx264.

    video_filter:
        None -> retain original resolution
        string -> apply given FFmpeg video filter
    """

    cmd = [
        FFMPEG,
        "-i", str(src),
    ]

    # --------------------------------------------------------
    # Apply scaling ONLY when a filter was supplied
    # --------------------------------------------------------

    if video_filter is not None:

        cmd.extend([
            "-vf", video_filter,
        ])

    # --------------------------------------------------------
    # Encoding options
    # --------------------------------------------------------

    cmd.extend([
        "-c:v", "libx264",
        "-preset", preset,
        "-crf", str(crf),

        # Widely compatible H.264 pixel format
        "-pix_fmt", "yuv420p",
    ])

    if overwrite:
        cmd.append("-y")

    cmd.append(str(dst))

    # Printable command
    cmd_str = " ".join(
        shlex.quote(str(c))
        for c in cmd
    )

    print("\n" + "=" * 90)

    print(f"[INPUT]   {src}")
    print(f"[OUTPUT]  {dst}")
    print(f"[CRF]     {crf}")

    if video_filter is not None:
        print(f"[FILTER]  {video_filter}")
    else:
        print("[FILTER]  None - original resolution retained")

    print(f"[COMMAND] {cmd_str}")

    # --------------------------------------------------------
    # Dry run
    # --------------------------------------------------------

    if DRY_RUN:

        print("[DRY RUN] FFmpeg not executed.")

        return True

    # --------------------------------------------------------
    # Execute
    # --------------------------------------------------------

    start_time = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Check result
    # --------------------------------------------------------

    if result.returncode != 0:

        print("\n[ERROR] FFmpeg failed.")

        print(f"Input:")
        print(src)

        print("\nFFmpeg error:")
        print(result.stderr)

        return False

    print(
        f"\n[OK] {dst.name}"
        f" | CRF={crf}"
        f" | Time={elapsed / 60:.2f} min"
        f" ({elapsed:.1f} s)"
    )

    return True


# ============================================================
# MAIN
# ============================================================

def main():

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    total = 0
    skipped = 0
    processed = 0
    failed = 0

    miniscope_count = 0
    optogenetics_count = 0
    other_count = 0

    print("=" * 90)
    print("TRAINING SESSION VIDEO COMPRESSION")
    print("=" * 90)

    print(f"\nInput root:")
    print(ROOT_DIR)

    print(f"\nOutput root:")
    print(OUTPUT_ROOT)

    print("\nSearching recursively for AVI files...")


    # ========================================================
    # RECURSIVE AVI SEARCH
    # ========================================================

    

    for avi in ROOT_DIR.rglob("*.avi"):

        if not avi.is_file():
            continue

        total += 1


        # ====================================================
        # SKIP ALREADY COMPRESSED SOURCE FILES
        # ====================================================

        if is_already_compressed_input(avi):

            print(
                f"\n[SKIP compressed input] {avi}"
            )

            skipped += 1

            continue


        # ====================================================
        # FIND DAY DIRECTORY
        # ====================================================

        day_path = find_day_path(avi)

        if day_path is None:

            print(
                f"\n[SKIP - no DayXX directory found] "
                f"{avi}"
            )

            skipped += 1

            continue

        day_name = day_path.name


        # ====================================================
        # MOUSE ID
        # ====================================================

        mouse_id = find_mouse_id_from_path(avi)


        # ====================================================
        # VIEW
        # ====================================================

        view = detect_view_from_path(avi)


        # ====================================================
        # VIDEO TYPE + COMPRESSION SETTINGS
        # ====================================================

        if is_optogenetic(avi):

            # ------------------------------------------------
            # OPTOGENETICS
            #
            # Keep existing resolution unchanged.
            # ------------------------------------------------

            video_type = "Optogenetics"

            crf = CRF_OPTOGENETICS

            video_filter = None

            optogenetics_count += 1


        elif is_miniscope(avi):

            # ------------------------------------------------
            # MINISCOPE
            #
            # Downsample to corresponding Optogenetics
            # resolution.
            #
            # Aspect ratio is preserved.
            # Any remaining space is padded.
            # ------------------------------------------------

            video_type = "Miniscope"

            crf = CRF_MINISCOPE

            video_filter = get_miniscope_filter(view)

            miniscope_count += 1


        else:

            # ------------------------------------------------
            # UNKNOWN / OTHER VIDEO TYPE
            #
            # Do not resize unknown videos.
            # Use normal CRF 22.
            # ------------------------------------------------

            video_type = "Other"

            crf = CRF_MINISCOPE

            video_filter = None

            other_count += 1


        # ====================================================
        # VIEW-SPECIFIC TRANSFORM
        # ====================================================
        #
        # Every TopView video must be rotated by 180 degrees.
        #
        # FrontView:
        #     no rotation
        #
        # TopView:
        #     hflip + vflip = 180-degree rotation
        #
        # For Miniscope TopView videos, this is appended after
        # the resize/pad filter.
        #
        # For Optogenetics / Other TopView videos, rotation is
        # the only video filter applied.
        # ====================================================

        video_filter = add_view_rotation_filter(
            video_filter=video_filter,
            view=view
        )


        # ====================================================
        # BUILD OUTPUT DIRECTORY
        # ====================================================
        #
        # Mirror everything from ROOT_DIR down to DayXX.
        #
        # Anything BELOW DayXX is intentionally removed.
        #
        #
        # Example:
        #
        # INPUT
        #
        # F:\trainingAggression_MR
        # └── Maladaptive
        #     └── Psilocybin
        #         └── 20260126_mouse1034814_trainingSessions
        #             └── Day03
        #                 └── Miniscope(FrontView)
        #                     └── 11-12-54.avi
        #
        #
        # OUTPUT
        #
        # F:\trainingSessions_compressedVideos
        # └── Maladaptive
        #     └── Psilocybin
        #         └── 20260126_mouse1034814_trainingSessions
        #             └── Day03
        #                 └── mouse1034814_Day03_frontView_comp.avi
        #
        # ====================================================

        try:

            relative_day_path = day_path.relative_to(
                ROOT_DIR
            )

        except ValueError:

            print(
                f"\n[ERROR] Day directory outside ROOT_DIR:"
            )

            print(day_path)

            failed += 1

            continue


        out_dir = (
            OUTPUT_ROOT
            / relative_day_path
        )

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ====================================================
        # OUTPUT FILE NAME
        # ====================================================

        out_name = (
            f"{mouse_id}_"
            f"{day_name}_"
            f"{view}_comp.avi"
        )

        out_path = (
            out_dir
            / out_name
        )


        # ====================================================
        # SKIP EXISTING OUTPUT
        # ====================================================

        if output_already_exists(out_path):

            print("\n" + "-" * 90)

            print(f"[SKIP existing] {out_path}")

            skipped += 1

            continue


        # ====================================================
        # DISPLAY DETECTED SETTINGS
        # ====================================================

        print("\n" + "-" * 90)

        print(f"[TYPE]    {video_type}")
        print(f"[MOUSE]   {mouse_id}")
        print(f"[DAY]     {day_name}")
        print(f"[VIEW]    {view}")

        if video_type == "Miniscope":

            if view == "frontView":

                print(
                    f"[RESIZE]  -> "
                    f"{FRONT_WIDTH}x{FRONT_HEIGHT}"
                )

            else:

                print(
                    f"[RESIZE]  -> "
                    f"{TOP_WIDTH}x{TOP_HEIGHT}"
                )

        else:

            print(
                "[RESIZE]  No resizing"
            )

        if view == "topView":

            print(
                "[ROTATE]  180 degrees"
            )

        else:

            print(
                "[ROTATE]  None"
            )


        # ====================================================
        # COMPRESS
        # ====================================================

        success = compress_with_ffmpeg(
            src=avi,
            dst=out_path,
            crf=crf,
            video_filter=video_filter
        )

        if success:

            processed += 1

        else:

            failed += 1


    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n")
    print("=" * 90)
    print("SUMMARY")
    print("=" * 90)

    print(
        f"Total AVI files found        : {total}"
    )

    print(
        f"Miniscope videos encountered : {miniscope_count}"
    )

    print(
        f"Optogenetics videos          : {optogenetics_count}"
    )

    print(
        f"Other videos                 : {other_count}"
    )

    print(
        f"Skipped                      : {skipped}"
    )

    print(
        f"Successfully compressed      : {processed}"
    )

    print(
        f"Failed                       : {failed}"
    )

    print("=" * 90)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

TRAINING SESSION VIDEO COMPRESSION

Input root:
A:\rawData\raw_trainingSessions\Maladaptive\Psilocybin\20260825_mouse1089289_trainingSessions\Day03

Output root:
C:\Users\stagk\OneDrive\Desktop\lol

Searching recursively for AVI files...

------------------------------------------------------------------------------------------
[TYPE]    Optogenetics
[MOUSE]   mouse1089289
[DAY]     Day03
[VIEW]    frontView
[RESIZE]  No resizing
[ROTATE]  None

[INPUT]   A:\rawData\raw_trainingSessions\Maladaptive\Psilocybin\20260825_mouse1089289_trainingSessions\Day03\Optogenetics(FrontView)\13-24-31.avi
[OUTPUT]  C:\Users\stagk\OneDrive\Desktop\lol\mouse1089289_Day03_frontView_comp.avi
[CRF]     16
[FILTER]  None - original resolution retained
[COMMAND] 'C:\Users\stagk\OneDrive\Desktop\ffmpeg\bin\ffmpeg.exe' -i 'A:\rawData\raw_trainingSessions\Maladaptive\Psilocybin\20260825_mouse1089289_trainingSessions\Day03\Optogenetics(FrontView)\13-24-31.avi' -c:v libx264 -preset slow -crf 16 -pix_fmt yuv420p -

In [3]:
from pathlib import Path
import cv2

# Compressed videos are written here
OUTPUT_ROOT = Path(
    r"C:\Users\Admin\Desktop\trainingAggression_MR\trainingSessions_compressedVideos"
)

MAX_DURATION_MINUTES = 20
MAX_DURATION_SECONDS = MAX_DURATION_MINUTES * 60

print(f"Checking AVI files in:\n{OUTPUT_ROOT}\n")

short_videos = []

# rglob searches this folder AND all subfolders
for video_path in OUTPUT_ROOT.rglob("*.avi"):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Could not open: {video_path}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    cap.release()

    if fps <= 0:
        print(f"Could not determine FPS: {video_path}")
        continue

    duration_seconds = frame_count / fps
    duration_minutes = duration_seconds / 60

    if duration_seconds < MAX_DURATION_SECONDS:
        short_videos.append((video_path, duration_minutes))


print("=" * 70)
print("SESSIONS WITH VIDEOS SHORTER THAN 20 MINUTES")
print("=" * 70)

if not short_videos:
    print("None found.")
else:
    for video_path, duration_minutes in short_videos:
        print(
            f"{video_path.name}  -->  "
            f"{duration_minutes:.2f} minutes\n"
            f"Session/folder: {video_path.parent.name}\n"
        )

print(f"\nTotal short videos found: {len(short_videos)}")

Checking AVI files in:
C:\Users\Admin\Desktop\trainingAggression_MR\trainingSessions_compressedVideos

SESSIONS WITH VIDEOS SHORTER THAN 20 MINUTES
mouse1013587_Day01_frontView_comp.avi  -->  12.74 minutes
Session/folder: Day01

mouse1013587_Day01_topView_comp.avi  -->  12.74 minutes
Session/folder: Day01

mouse1106648_Day05_frontView_comp.avi  -->  0.00 minutes
Session/folder: Day05

mouse1079919_Day02_frontView_comp.avi  -->  17.33 minutes
Session/folder: Day02

mouse1079919_Day02_topView_comp.avi  -->  17.33 minutes
Session/folder: Day02


Total short videos found: 5


In [10]:
from pathlib import Path
import subprocess

INPUT_FOLDER = Path(r"A:\rawData\compressedVideos_AVI\trainingSessions_DLC\Maladaptive\Psilocybin&Stim\20260818_mouse1106648_trainingSessions\Day16")
OUTPUT_FOLDER = Path(r"C:\Users\stagk\OneDrive\Desktop\lol\lol")

for video in INPUT_FOLDER.rglob("*"):
    if not video.is_file() or video.suffix.lower() != ".avi":
        continue

    # Preserve the directory structure relative to INPUT_FOLDER
    relative_path = video.relative_to(INPUT_FOLDER)
    output = OUTPUT_FOLDER / relative_path

    # Create output subdirectories if necessary
    output.parent.mkdir(parents=True, exist_ok=True)

    print(f"Processing: {video}")
    print(f"       -> : {output}")

    subprocess.run([
        r"C:\Users\stagk\OneDrive\Desktop\ffmpeg\bin\ffmpeg.exe",
        "-y",
        "-i", str(video),
        "-ss", "00:00:20",
        "-c", "copy",
        str(output),
    ], check=True)

print("Done.")

Processing: A:\rawData\compressedVideos_AVI\trainingSessions_DLC\Maladaptive\Psilocybin&Stim\20260818_mouse1106648_trainingSessions\Day16\mouse1106648_Day16_frontView_comp.avi
       -> : C:\Users\stagk\OneDrive\Desktop\lol\lol\mouse1106648_Day16_frontView_comp.avi
Processing: A:\rawData\compressedVideos_AVI\trainingSessions_DLC\Maladaptive\Psilocybin&Stim\20260818_mouse1106648_trainingSessions\Day16\mouse1106648_Day16_topView_comp.avi
       -> : C:\Users\stagk\OneDrive\Desktop\lol\lol\mouse1106648_Day16_topView_comp.avi
Done.
